## 触发词检测

欢迎来到第三周的第二个也是最后一个编程作业！ 

在本周的视频中，你了解了如何将深度学习应用于语音识别。在此作业中，你将构建语音数据集并实现触发词检测算法（有时也称为关键字检测或唤醒词检测）。 

* 触发词检测是一项允许 Amazon Alexa、Google Home、Apple Siri 和百度 DuerOS 等设备在听到某个词时唤醒的技术。  
* 对于本练习，我们的触发词将是“activate”。每次听到你说“activate”时，它都会发出“叮叮”声。 
* 在本作业结束时，你将能够录制自己说话的片段，并让算法在检测到你说“activate”时触发提示音。 
* 完成此作业后，也许你还可以将其扩展为在你的Notebook电脑上运行，以便每次你说“activate”时，它都会启动你最喜欢的应用程序，或者打开你家里的网络连接灯，或者触发一些其他事件？ 

<img src="images/sound.png" style="width:1000px;height:150px;">

在本作业中，你将学习： 
- 构建语音识别项目
- 合成和处理录音以创建训练/开发数据集
- 训练触发词检测模型并进行预测

让我们开始吧！

## 关于提交至 AutoGrader 的重要说明

向 AutoGrader 提交作业前，请确认满足以下要求：

1. 未在作业中添加 _额外的_  `print` 语句。
2. 未在作业中添加 _额外的_ 代码单元格。
3. 未修改任何函数的参数。
4. 未在计分练习中使用全局变量。除非明确要求使用全局变量，否则应使用局部变量。
5. 未修改无需改动的作业代码，例如创建 _额外的_ 变量。

如果违反上述要求，提交作业时可能出现 `Grader Error: Grader feedback not found` 或其他意外错误。在求助或调试作业之前，请先检查这些情况。如果不记得自己做过哪些改动，可以按照 [说明](https://www.coursera.org/learn/nlp-sequence-models/supplement/qHIve/h-ow-to-refresh-your-workspace) 获取一份新的作业。

## 目录

- [依赖库](#0)
- [1 - 数据合成：创建语音数据集](#1)
    - [1.1 - 聆听数据](#1-1)
    - [1.2 - 从录音到频谱图](#1-2)
    - [1.3 - 生成单个训练样本](#1-3)
        - [练习 1 - is_overlapping](#ex-1)
        - [练习 2 - insert_audio_clip](#ex-2)
        - [练习 3 - insert_ones](#ex-3)
        - [练习 4 - create_training_example](#ex-4)
    - [1.4 - 完整训练集](#1-4)
    - [1.5 - 开发集](#1-5)
- [2 - 模型](#2)
    - [2.1 - 构建模型](#2-1)
        - [练习 5 - modelf](#ex-5)
    - [2.2 - 拟合模型](#2-2)
    - [2.3 - 测试模型](#2-3)
- [3 - 做出预测](#3)
    - [3.1 - 开发示例测试](#3-1)
- [4 - 尝试你自己的例子！ (OPTIONAL/UNGRADED)](#4)

<a name='0'></a>
## 依赖库

In [ ]:
### v2.2

In [ ]:
import numpy as np
from pydub import AudioSegment
import random
import sys
import io
import os
import glob
import IPython
from td_utils import *
%matplotlib inline

<a name='1'></a>
## 1 - 数据合成：创建语音数据集 

让我们首先为触发词检测算法构建数据集。 
* 理想情况下，语音数据集应尽可能接近你想要运行它的应用程序。 
* 在这种情况下，你希望在工作环境（图书馆、家庭、办公室、开放空间...）中检测“activate”一词。 
* 因此，你需要在不同的背景声音下混合积极词（“activate”）和消极词（除了激活之外的随机词）来创建录音。让我们看看如何创建这样的数据集。 

<a name='1-1'></a>
### 1.1 - 聆听数据   

* 你的一位朋友正在帮助你完成这个项目，他们去了整个地区的图书馆、咖啡馆、餐馆、家庭和办公室记录背景噪音，以及人们说积极/消极话语的音频片段。该数据集包括使用各种口音的人。 
* 在raw_data目录中，你可以找到正面词、负面词和背景噪音的原始音频文件的子集。你将使用这些音频文件来合成数据集来训练模型。 
    * “activate”目录包含人们说“activate”这个词的正面例子。 
    * “负面”目录包含人们随意说出“activate”以外的单词的负面例子。 
    * 每个录音只有一个单词。 
    * “backgrounds”目录包含不同环境下的10秒背景噪音片段。

运行下面的单元格来听一些示例。

In [ ]:
IPython.display.Audio("./raw_data/activates/1.wav")

In [ ]:
IPython.display.Audio("./raw_data/negatives/4.wav")

In [ ]:
IPython.display.Audio("./raw_data/backgrounds/1.wav")

你将使用这三种类型的录音（正片/负片/背景）来创建带标签的数据集。

<a name='1-2'></a>
### 1.2 - 从录音到频谱图

录音到底是什么？ 
* 麦克风会记录气压随时间的微小变化，而你的耳朵也将这些微小的气压变化感知为声音。 
* 你可以将录音视为一长串数字，用于测量麦克风检测到的微小气压变化。 
* 我们将使用以 44100 Hz（或 44100 Hertz）采样的音频。 
    * 这意味着麦克风每秒为我们提供 44,100 个数字。 
    * 因此，10 秒的音频片段由 441,000 个数字表示（= $10 \times 44,100$）。 

#### 频谱图
* 从这种“原始”音频表示中很难弄清楚是否说出了“activate”一词。 
* 为了帮助你的序列模型更轻松地学习检测触发词，我们将计算音频的*频谱图*。 
* 频谱图告诉我们在任何时刻音频片段中存在多少不同的频率。 
* 如果你曾经上过信号处理或傅里叶变换的高级课程：
    * 声谱图是通过在原始音频信号上滑动窗口并使用傅里叶变换计算每个窗口中最活跃的频率来计算的。 
    * 如果你不明白前一句，请不要担心。

让我们看一个例子。 

In [ ]:
IPython.display.Audio("audio_examples/example_train.wav")

In [ ]:
x = graph_spectrogram("audio_examples/example_train.wav")

上图表示每个频率（y 轴）在多个时间步（x 轴）上的活跃程度。 

<img src="images/spectrogram.png" style="width:500px;height:200px;">
<center><strong>图 1</strong>：录音的频谱图</center>


* 频谱图中的颜色显示了不同时间点音频中不同频率出现（响亮）的程度。 
* 绿色表示音频片段中某个频率更活跃或更多（声音更大）。
* 蓝色方块表示不太活跃的频率。
* 输出频谱图的维度取决于频谱图软件的超参数和输入的长度。 
* 在此 Notebook 中，我们将使用 10 秒音频片段作为训练样本的“标准长度”。 
    * 频谱图的时间步数将为 5511。 
    * 稍后你将看到频谱图将是网络的输入 $x$，因此 $T_x = 5511$。

In [ ]:
_, data = wavfile.read("audio_examples/example_train.wav")
print("Time steps in audio recording before spectrogram", data[:,0].shape)
print("Time steps in input after spectrogram", x.shape)

现在，你可以定义：

In [ ]:
Tx = 5511 # 从频谱图中输入到模型的时间步数
n_freq = 101 # 频谱图每个时间步输入模型的频率数

#### 划分时间间隔
请注意，我们可以将 10 秒的时间间隔划分为不同的单位（步长）。
* 原始音频将 10 秒分为 441,000 个单位。
* 频谱图将 10 秒分为 5,511 个单位。
    * $T_x = 5511$
* 你将使用Python模块`pydub`来合成音频，它将10秒分为10,000个单位。
* 我们模型的输出将把 10 秒分为 1,375 个单位。
    * $T_y = 1375$
    * 对于 1375 个时间步中的每一个，模型都会预测某人最近是否说完触发词“activate”。 
* 所有这些都是超参数，可以更改（除了 441000，它是麦克风的函数）。 
* 我们选择了用于语音系统的标准范围内的值。

In [ ]:
Ty = 1375 # 我们模型输出中的时间步数

<a name='1-3'></a>
### 1.3 - 生成单个训练样本

#### 合成数据的优势
由于语音数据很难获取和标记，因此你将使用激活、负片和背景的音频片段来合成训练数据。 
* 录制大量带有随机“activate”的 10 秒音频片段是相当慢的。 
* 相反，记录大量积极和消极的词语，并单独记录背景噪音（或从免费的在线资源下载背景噪音）会更容易。 

#### 合成音频片段的流程
* 要合成单个训练样本，你将：
    - 随机选择一个 10 秒背景音频片段
    - 在这10秒内随机插入0-4个“activate”音频片段。片段
    - 在这10秒内随机插入0-2个负面词语的音频片段。片段
* 因为你已将“activate”一词合成到背景片段中，所以你可以准确地知道“activate”在 10 秒片段中何时出现。 
    * 稍后你会看到，这也使得生成标签 $y^{\langle t \rangle}$ 变得更加容易。 

#### Pydub
* 你将使用 pydub 包来操作音频。 
* Pydub 将原始音频文件转换为 Pydub 数据结构列表。
    * 不要担心数据结构的细节。
* Pydub 使用 1ms 作为离散化间隔（1 ms 为 1 毫秒 = 1/1000 秒）。
    * 这就是为什么 10 秒的片段总是使用 10,000 个步骤来表示。 

In [ ]:
# 使用 pydub 加载音频片段 
activates, negatives, backgrounds = load_raw_audio('./raw_data/')

print("background len should be 10,000, since it is a 10 sec clip\n" + str(len(backgrounds[0])),"\n")
print("activate[0] len may be around 1000, since an `activate` audio clip is usually around 1 second (but varies a lot) \n" + str(len(activates[0])),"\n")
print("activate[1] len: different `activate` clips can have different lengths\n" + str(len(activates[1])),"\n")

#### 在背景音频之上叠加正/负“单词”音频片段

* 给定一个 10 秒的背景片段和一个包含正面或负面单词的短音频片段，你需要能够将单词音频片段“添加”到背景音频之上。
* 你将在背景中插入多个正面/负面单词的片段，并且你不想在与你之前添加的另一个片段重叠的位置插入“activate”或随机单词。 
    * 为了确保“单词”音频片段在插入时不会重叠，你将跟踪先前插入的音频片段的时间。 
* 需要明确的是，当你将 1 秒的“activate”插入到 10 秒的咖啡馆噪音片段中时，**你最终不会得到 11 秒的片段。** 
    * 生成的音频片段仍然有 10 秒长。
    * 稍后你将看到 pydub 如何允许你执行此操作。 

#### 标注正样本和负样本词
* 回想一下，标签 $y^{\langle t \rangle}$ 代表某人是否刚刚说完“activate”。 
    * $y^{\langle t \rangle} = 1$ 当该片段说完“activate”时。
    * 给定一个背景片段，我们可以为所有 $t$ 初始化 $y^{\langle t \rangle}=0$，因为该片段不包含任何“activate”。 
* 当你插入或覆盖“activate”片段时，你还将更新 $y^{\langle t \rangle}$ 的标签。
    * 我们将更新输出的 50 个步骤以获得目标标签 1，而不是更新单个时间步的标签。 
    * 回想一下触发词检测的讲座，更新几个连续的时间步可以使训练数据更加平衡。
* 你将训练一个 GRU（门控循环单元）来检测某人何时**完成**说“activate”。 

##### 示例
* 假设合成的“activate”片段在 10 秒音频中的 5 秒标记处结束 - 恰好进入片段的中间。 
* 回想一下 $T_y = 1375$，因此时间步 $687 = $ `int(1375*0.5)` 对应于音频片段中 5 秒的时刻。 
* 设置$y^{\langle 688 \rangle} = 1$。 
* 我们允许 GRU 在此时刻**之后**的一个短时间区间内检测到“activate”，因此实际会将标签 $y^{\langle t \rangle}$ 的**连续 50 个值**设为 1。 
    * 具体来说，我们有 $y^{\langle 688 \rangle} = y^{\langle 689 \rangle} = \cdots = y^{\langle 737 \rangle} = 1$。  

##### 合成数据更容易标注
* 这是合成训练数据的另一个原因：如上所述，生成这些标签 $y^{\langle t \rangle}$ 相对简单。 
* 相比之下，如果你在麦克风上录制了 10 秒的音频，那么对于一个人来说，聆听它并准确地手动标记“activate”完成的时间是相当耗时的。 

#### 可视化标签
* 这是一个说明片段中标签 $y^{\langle t \rangle}$ 的图。
    * 我们插入了“activate”、“无辜”、“activate”、“宝贝”。 
    * 请注意，积极标签“1”仅与积极词相关联。 

<img src="images/label_diagram.png" style="width:500px;height:200px;">
<center> <strong>图 2</strong> </center>



#### 辅助函数

要实现训练集综合过程，你将使用以下辅助函数。 
* 所有这些函数都将使用 1ms 离散化间隔
* 10 秒的音频始终离散化为 10,000 个步骤。 


1. `get_random_time_segment(segment_ms)`
    * 从背景音频中检索随机时间段。
2. `is_overlapping(segment_time, existing_segments)`
    * 检查时间段是否与现有段重叠
3. `insert_audio_clip(background, audio_clip, existing_times)`
    * 在背景音频中随机插入音频片段
    * 使用函数 `get_random_time_segment` 和 `is_overlapping`
4. `insert_ones(y, segment_end_ms)`
    * 在单词“activate”之后在标签向量 y 中插入额外的 1

#### 获取随机时间段

* 函数 `get_random_time_segment(segment_ms)` 返回一个随机时间段，我们可以在其中插入持续时间为 `segment_ms` 的音频片段。 
* 请通读代码以确保你理解它在做什么。 

In [ ]:
def get_random_time_segment(segment_ms):
    """
    在长度为 10,000 ms 的音频片段中，随机选取一个时长为 segment_ms 的时间段。
    
    参数：
    segment_ms -- 音频片段的持续时间（以毫秒为单位）（"ms" 代表 "milliseconds"）
    
    返回：
    segment_time -- (segment_start, segment_end) 的元组（以毫秒为单位）
    """
    
    segment_start = np.random.randint(low=0, high=10000-segment_ms)   # 确保片段不会超过 10 秒背景 
    segment_end = segment_start + segment_ms - 1
    
    return (segment_start, segment_end)

#### 检查音频片段是否重叠

* 假设你已在段 (1000,1800) 和 (3400,4500) 处插入音频片段。
    * 第一段从步骤 1000 开始，到步骤 1800 结束。 
    * 第二个时间段从 3400 开始，到 4500 结束。
* 如果我们正在考虑是否在 (3000,3600) 处插入一个新的音频片段，这是否与之前插入的片段之一重叠？ 
    * 在这种情况下，(3000,3600) 与 (3400,4500) 重叠，因此不应在此处插入音频片段。
* 出于此函数的目的，将 (100,200) 和 (200,250) 定义为重叠，因为它们在时间步 200 处重叠。 
* (100,199) 与 (200,250) 不重叠。 

<a name='ex-1'></a>
### 练习 1 - is_overlapping

* 实现 `is_overlapping(segment_time, existing_segments)`，检查新的时间段是否与已有时间段重叠。 
* 你需要执行 2 个步骤：

1. 创建一个“False”标志，如果你发现存在重叠，稍后将其设置为“True”。
2. 遍历 previous_segments 的起止时间，将其与新时间段的起止时间进行比较。如果存在重叠，则将步骤 (1) 中定义的标志设为 True。 

你可以使用：
```python
for ....:
        if ... <= ... and ... >= ...:
            ...
```
提示：满足以下条件时存在重叠：
* 新片段在上一个片段结束之前开始**和**
* 新片段在上一个片段开始后结束。

In [ ]:
# UNQ_C1
# GRADED FUNCTION: is_overlapping

def is_overlapping(segment_time, previous_segments):
    """
    检查一个时间段是否与已有时间段重叠。
    
    参数：
    segment_time -- 新段的 (segment_start, segment_end) 的元组
    previous_segments -- 现有段的 (segment_start, segment_end) 元组列表
    
    返回：
    如果时间段与任何现有段重叠，则为 True，否则为 False
    """
    
    segment_start, segment_end = segment_time
    
    ### START CODE HERE ### (≈ 4 lines)
    # 步骤 1：将重叠初始化为 "False" 标志。 （≈ 1 行）
    overlap = None
    
    # 第 2 步：遍历 previous_segments 的起止时间。
    # 比较起止时间，如果存在重叠，将标志设为 True（约 3 行代码）
    for previous_start, previous_end in previous_segments:
        if None:
            overlap = None
            break
    ### END CODE HERE ###

    return overlap

In [ ]:
### 该单元格不可编辑

# 单元测试
def is_overlapping_test(target):
    assert target((670, 1430), []) == False, "Overlap with an empty list must be False"
    assert target((500, 1000), [(100, 499), (1001, 1100)]) == False, "Almost overlap, but still False"
    assert target((750, 900), [(100, 750), (1001, 1100)]) == True, "Must overlap with the end of first segment"
    assert target((750, 1250), [(300, 600), (1250, 1500)]) == True, "Must overlap with the beginning of second segment"
    assert target((750, 1250), [(300, 600), (600, 1500), (1600, 1800)]) == True, "Is contained in second segment"
    assert target((800, 1100), [(300, 600), (900, 1000), (1600, 1800)]) == True, "New segment contains the second segment"

    print("\033[92m All tests passed!")
    
is_overlapping_test(is_overlapping)

In [ ]:
overlap1 = is_overlapping((950, 1430), [(2000, 2550), (260, 949)])
overlap2 = is_overlapping((2305, 2950), [(824, 1532), (1900, 2305), (3424, 3656)])
print("Overlap 1 = ", overlap1)
print("Overlap 2 = ", overlap2)

**预期输出**：

<table>
    <tr>
        <td>
            <strong>重叠 1</strong>
        </td>
        <td>
           错误
        </td>
    </tr>
    <tr>
        <td>
            <strong>重叠2</strong>
        </td>
        <td>
           正确
        </td>
    </tr>
</table>

#### 插入音频片段

* 使用前面的辅助函数，在 10 秒背景音频中的随机位置插入一个新音频片段。
* 我们将确保任何新插入的片段不会与之前插入的片段重叠。 

<a name='ex-2'></a>
### 练习 2 - insert_audio_clip
* 实现 `insert_audio_clip()` 将音频片段叠加到背景 10 秒片段上。 
* 你实现 4 个步骤：

1. 获取待插入音频片段的时长。
    * 随机选择一个时间段，其时长与待插入音频片段的时长相同。
2. 确保该时间段不与之前的任何时间段重叠。 
    * 如果重叠，则返回步骤1并选择新的时间段。
3. 将新时间段追加到现有时间段列表中
    * 这会跟踪你插入的所有片段。  
4. 使用 pydub 将音频片段覆盖在背景上。我们已经为你实现了这一点。

In [ ]:
# UNQ_C2
# GRADED FUNCTION: insert_audio_clip

def insert_audio_clip(background, audio_clip, previous_segments):
    """
    以随机时间步在背景噪声上插入新的音频片段，确保 
    音频片段不与现有片段重叠。
    
    参数：
    background -- 10秒的背景录音。  
    audio_clip -- 要插入/覆盖的音频片段。 
    previous_segments -- 已放置音频片段的时间
    
    返回：
    new_background -- 更新的背景音频
    """
    
    # 获取音频片段的持续时间（以毫秒为单位）
    segment_ms = len(audio_clip)
    
    ### START CODE HERE ### 
    # 步骤 1：使用其中一个辅助函数选择要插入的随机时间段 
    # 新的音频片段。 （≈ 1 行）
    segment_time = None
    
    # 步骤2：检查新的segment_time是否与其中一个previous_segments重叠。如果是这样，请保留 
    # 随机挑选新的segment_time，直到不重叠为止。为了避免无限循环
    # 我们重试 5 times(≈ 2 lines)
    retry = 5 
    while None and retry >= 0:
        segment_time = None
        retry = retry - 1
    ### END CODE HERE ###
        #print(segment_time)
    # 如果最后一次尝试不重叠，则将其插入背景
    if not is_overlapping(segment_time, previous_segments):
    ### START CODE HERE ### 
        # 步骤 3：将新的 segment_time 追加到 previous_segments 列表中（≈ 1 行）
        None
    ### END CODE HERE ###
        # 步骤4：叠加音频片段和背景
        new_background = background.overlay(audio_clip, position = segment_time[0])
    else:
        #print("Timeouted")
        new_background = background
        segment_time = (10000, 10000)
    
    return new_background, segment_time

In [ ]:
### 该单元格不可编辑

# 单元测试
def insert_audio_clip_test(target):
    np.random.seed(5)
    audio_clip, segment_time = target(backgrounds[0], activates[0], [(0, 4400)])
    duration = segment_time[1] - segment_time[0]
    #print(f"xx: {segment_time}")
    assert segment_time[0] > 4400, "Error: The audio clip is overlapping with the first segment"
    assert duration + 1 == len(activates[0]) , "The segment length must match the audio clip length"
    assert audio_clip != backgrounds[0] , "The audio clip must be different than the pure background"
    assert segment_time == (7286, 8201), f"Wrong segment. Expected: (7286, 8201) got:{segment_time}"
    
    # 无法将片段插入背景
    audio_clip, segment_time = target(backgrounds[0], activates[0], [(0, 9999)])
    assert segment_time == (10000, 10000), "Segment must match the out by max-retry mark"
    assert audio_clip == backgrounds[0], "output audio clip must be exactly the same input background"
    
    print("\033[92m All tests passed!")

insert_audio_clip_test(insert_audio_clip)

In [ ]:
np.random.seed(5)
audio_clip, segment_time = insert_audio_clip(backgrounds[0], activates[0], [(3790, 4400)])
audio_clip.export("insert_test.wav", format="wav")
print("Segment Time: ", segment_time)
IPython.display.Audio("insert_test.wav")

**预期输出**

<table>
    <tr>
        <td>
            <strong>分段时间</strong>
        </td>
        <td>
           (2254, 3169)
        </td>
    </tr>
</table>

In [ ]:
# 预期音频
IPython.display.Audio("audio_examples/insert_reference.wav")

#### 在正样本目标对应的标签位置插入 1

* 实现代码来更新标签 $y^{\langle t \rangle}$，假设你刚刚插入了“activate”音频片段。
* 在下面的代码中，`y` 是一个 `(1,1375)` 维度向量，因为 $T_y = 1375$ 。 
* 如果“activate”音频片段在时间步 $t$ 结束，则设置 $y^{\langle t+1 \rangle} = 1$，并将随后连续的 49 个值也设为 1。
    * 请注意，如果目标单词出现在整个音频片段的末尾附近，则可能没有 50 个额外的时间步可以设置为 1。
    * 注意不要越过数组边界而尝试更新 `y[0][1375]`。由于 $T_y = 1375$，有效索引范围是 `y[0][0]` 至 `y[0][1374]`。 
    * 所以如果“activate”在步骤1370结束，你只会得到设置的`y[0][1371] = y[0][1372] = y[0][1373] = y[0][1374] = 1`

<a name='ex-3'></a>
### 练习 3 - insert_ones 
实现`insert_ones()`。 
* 你可以使用 for 循环。 
* 如果你想使用Python的数组切片操作，你也可以这样做。
* 如果段结束于 `segment_end_ms`（使用 10000 步离散化），
    * 要将其转换为输出 $y$ 的索引（使用 $1375$ 步离散化），我们将使用以下公式：  
```
    segment_end_y = int(segment_end_ms * Ty / 10000.0)
```

In [ ]:
# UNQ_C3
# GRADED FUNCTION: insert_ones

def insert_ones(y, segment_end_ms):
    """
    更新标签向量 y。严格在段结束之后的 50 个输出步骤的标签 
    应设置为 1。严格来说，我们的意思是 segment_end_y 的标签应为 0，而
    以下 50 个标签应该是。
    
    
    参数：
    y -- NumPy 形状为 (1, Ty) 的数组，训练样本的标签
    segment_end_ms -- 该段的结束时间（以毫秒为单位）
    
    返回：
    y -- 更新标签
    """
    _, Ty = y.shape
    
    # 背景持续时间（以频谱图时间步表示）
    segment_end_y = int(segment_end_ms * Ty / 10000.0)
    
    if segment_end_y < Ty:
        # 将背景标签中的正确索引加 1 (y)
        ### START CODE HERE ### (≈ 3 lines)
        for i in range(None, None):
            if None < None:
                y[0, i] = None
        ### END CODE HERE ###
    
    return y

In [ ]:
### 该单元格不可编辑

# 单元测试
import random
def insert_ones_test(target):
    segment_end_y = random.randrange(0, Ty - 50) 
    segment_end_ms = int(segment_end_y * 10000.4) / Ty;    
    arr1 = target(np.zeros((1, Ty)), segment_end_ms)

    assert type(arr1) == np.ndarray, "Wrong type. Output must be a numpy array"
    assert arr1.shape == (1, Ty), "Wrong shape. It must match the input shape"
    assert np.sum(arr1) == 50, "It must insert exactly 50 ones"
    assert arr1[0][segment_end_y - 1] == 0, f"Array at {segment_end_y - 1} must be 0"
    assert arr1[0][segment_end_y] == 0, f"Array at {segment_end_y} must be 0"
    assert arr1[0][segment_end_y + 1] == 1, f"Array at {segment_end_y + 1} must be 1"
    assert arr1[0][segment_end_y + 50] == 1, f"Array at {segment_end_y + 50} must be 1"
    assert arr1[0][segment_end_y + 51] == 0, f"Array at {segment_end_y + 51} must be 0"
    
    arr1 = target(np.zeros((1, Ty)), 9632)
    assert np.sum(arr1) == 50, f"Expected sum of 50, but got {np.sum(arr1)}"
    arr1 = target(np.zeros((1, Ty)), 9637)
    assert np.sum(arr1) == 49, f"Expected sum of 49, but got {np.sum(arr1)}"
    arr1 = target(np.zeros((1, Ty)), 10008)
    assert np.sum(arr1) == 0, f"Expected sum of 0, but got {np.sum(arr1)}"
    arr1 = target(np.zeros((1, Ty)), 10000)
    assert np.sum(arr1) == 0, f"Expected sum of 0, but got {np.sum(arr1)}"
    arr1 = target(np.zeros((1, Ty)), 9996)
    assert np.sum(arr1) == 0, f"Expected sum of 0, but got {np.sum(arr1)}"
    arr1 = target(np.zeros((1, Ty)), 9990)
    assert np.sum(arr1) == 1, f"Expected sum of 1, but got {np.sum(arr1)}"
    arr1 = target(np.zeros((1, Ty)), 9980)
    assert np.sum(arr1) == 2, f"Expected sum of 2, but got {np.sum(arr1)}"

    print("\033[92m All tests passed!")
    
insert_ones_test(insert_ones)

In [ ]:
arr1 = insert_ones(np.zeros((1, Ty)), 9700)
plt.plot(insert_ones(arr1, 4251)[0,:])
print("sanity checks:", arr1[0][1333], arr1[0][634], arr1[0][635])

**预期输出**
<table>
    <tr>
        <td>
            <strong>健全性检查</strong>：
        </td>
        <td>
           0.0 1.0 0.0
        </td>
    </tr>
</table>
<img src="images/ones_reference.png" style="width:320px;height:240px;">

#### 创建训练样本
最后，你可以使用`insert_audio_clip`和`insert_ones`创建一个新的训练样本。

<a name='ex-4'></a>
### 练习 4 - create_training_example

实现`create_training_example()`。你需要执行以下步骤：

1. 将标签向量 $y$ 初始化为 NumPy 零数组和形状 $(1, T_y)$。
2. 将现有段集初始化为空列表。
3. 随机选择0到4个“activate”音频片段，并将它们插入到10秒的片段中。还将标签插入标签向量 $y$ 中的正确位置。
4. 随机选择0到2个负片音频片段，插入到10秒片段中。 


In [ ]:
# UNQ_C4
# GRADED FUNCTION: create_training_example

def create_training_example(background, activates, negatives, Ty):
    """
    创建具有给定背景、激活和底片的训练样本。
    
    参数：
    background -- 10秒的背景录音
    activates -- 单词 "activate" 的音频片段列表
    negatives -- 非 "activate" 的随机单词的音频片段列表
    Ty -- 输出中的时间步数

    返回：
    x -- 训练样本的频谱图
    y -- 频谱图每个时间步的标签
    """
    
    # 让背景更安静
    background = background - 20

    ### START CODE HERE ###
    # 步骤 1：初始化零的 y（标签向量）（≈ 1 行）
    y = None

    # 步骤 2：将分段时间初始化为空列表（≈ 1 行）
    previous_segments = None
    ### END CODE HERE ###
    
    # 从 "activates" 录音的整个列表中选择 0-4 个随机 "activate" 音频片段
    number_of_activates = np.random.randint(0, 5)
    random_indices = np.random.randint(len(activates), size=number_of_activates)
    random_activates = [activates[i] for i in random_indices]
    
    ### START CODE HERE ### (≈ 3 lines)
    # 步骤 3：循环播放随机选择的 "activate" 片段并插入背景
    for one_random_activate in random_activates:
        # 将音频片段插入背景
        background, segment_time = None
        # 从 segment_time 中检索 segment_start 和 segment_end
        segment_start, segment_end = None
        # 在 "y" 的 segment_end 处插入标签
        y = None
    ### END CODE HERE ###

    # 从 "negatives" 录音的整个列表中选择 0-2 个随机底片录音
    number_of_negatives = np.random.randint(0, 3)
    random_indices = np.random.randint(len(negatives), size=number_of_negatives)
    random_negatives = [negatives[i] for i in random_indices]

    ### START CODE HERE ### (≈ 2 lines)
    # 步骤 4：循环播放随机选择的负片片段并插入背景
    for random_negative in random_negatives:
        # 将音频片段插入背景 
        background, _ = None
    ### END CODE HERE ###
    
    # 将音频片段的音量统一到目标水平
    background = match_target_amplitude(background, -20.0)

    # 导出新训练样本 
    file_handle = background.export("train" + ".wav", format="wav")
    
    # 获取并绘制新录音的频谱图（正负叠加背景）
    x = graph_spectrogram("train.wav")
    
    return x, y

In [ ]:
### 该单元格不可编辑

# 单元测试
def create_training_example_test(target):
    np.random.seed(18)
    x, y = target(backgrounds[0], activates, negatives, 1375)
    
    assert type(x) == np.ndarray, "Wrong type for x"
    assert type(y) == np.ndarray, "Wrong type for y"
    assert tuple(x.shape) == (101, 5511), "Wrong shape for x"
    assert tuple(y.shape) == (1, 1375), "Wrong shape for y"
    assert np.all(x > 0), "All x values must be higher than 0"
    assert np.all(y >= 0), "All y values must be higher or equal than 0"
    assert np.all(y <= 1), "All y values must be smaller or equal than 1"
    assert np.sum(y) >= 50, "It must contain at least one activate"
    assert np.sum(y) % 50 == 0, "Sum of activate marks must be a multiple of 50"
    assert np.isclose(np.linalg.norm(x), 39745552.52075), "Spectrogram is wrong. Check the parameters passed to the insert_audio_clip function"

    print("\033[92m All tests passed!")

create_training_example_test(create_training_example)

In [ ]:
# 设置随机种子
np.random.seed(18)
x, y = create_training_example(backgrounds[0], activates, negatives, Ty)

**预期输出**
<img src="images/train_reference.png" style="width:320px;height:240px;">

现在你可以聆听你创建的训练样本并将其与上面生成的频谱图进行比较。

In [ ]:
IPython.display.Audio("train.wav")

**预期输出**

In [ ]:
IPython.display.Audio("audio_examples/train_reference.wav")

最后，你可以为生成的训练样本绘制关联标签。

In [ ]:
plt.plot(y[0])

**预期输出**
<img src="images/train_label.png" style="width:320px;height:240px;">

<a name='1-4'></a>
### 1.4 - 完整训练集

* 你现在已经实现了生成单个训练样本所需的代码。 
* 我们使用这个过程生成了一个大的训练集。 
* 为了节省时间，我们生成了一个包含 32 个示例的较小的训练集。 

In [ ]:
# 开始跳过评分
np.random.seed(4543)
nsamples = 32
X = []
Y = []
for i in range(0, nsamples):
    if i%10 == 0:
        print(i)
    x, y = create_training_example(backgrounds[i % 2], activates, negatives, Ty)
    X.append(x.swapaxes(0,1))
    Y.append(y.swapaxes(0,1))
X = np.array(X)
Y = np.array(Y)
# 评分结束跳过

你希望将数据集保存到一个文件中，以便稍后在更现实的环境中工作时加载。我们让你参考以下代码。不要尝试在 Coursera 中运行它，因为文件系统是只读的，并且你无法保存文件。

In [ ]:
# 保存数据以供进一步使用
# np.save(f'./XY_train/X.npy', X)
# np.save(f'./XY_train/Y.npy', Y)
# 加载预处理后的训练样本
# X = np.load("./XY_train/X.npy")
# Y = np.load("./XY_train/Y.npy")

<a name='1-5'></a>
### 1.5 - 开发集

* 为了测试我们的模型，我们记录了一组包含 25 个示例的开发集。 
* 虽然训练数据是合成的，但开发集应采用与实际输入相同的数据分布。 
* 因此，我们录制了 25 个 10 秒的音频片段，其中人们说“activate”和其他随机单词，并手动标记它们。 
* 这遵循课程 3“构建机器学习项目”中描述的原则，即我们应该创建尽可能类似于测试集分布的开发集
    * 这就是为什么我们的**开发集使用真实音频**而不是合成音频。 


In [ ]:
# 加载预处理的开发集示例
X_dev = np.load("./XY_dev/X_dev.npy")
Y_dev = np.load("./XY_dev/Y_dev.npy")

<a name='2'></a>
## 2 - 模型

* 现在你已经构建了数据集，让我们编写并训练触发词检测模型！ 
* 该模型将使用一维卷积层、GRU 层和全连接层。 
* 让我们加载允许你在 Keras 中使用这些层的包。加载可能需要一分钟时间。 

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.models import Model, load_model, Sequential
from tensorflow.keras.layers import Dense, Activation, Dropout, Input, Masking, TimeDistributed, LSTM, Conv1D
from tensorflow.keras.layers import GRU, Bidirectional, BatchNormalization, Reshape
from tensorflow.keras.optimizers import Adam

<a name='2-1'></a>
### 2.1 - 构建模型

我们的目标是建立一个网络，该网络将摄取频谱图并在检测到触发词时输出信号。该网络将使用 4 层：
    * 卷积层
    * 两个 GRU 层
    * 全连接层。 

下面是将要使用的网络架构。

<img src="images/model.png" style="width:600px;height:600px;">
<center> <strong>图 3</strong> </center>

##### 一维卷积层
该模型的一个关键层是一维卷积步骤（靠近图 3 底部）。 
* 输入5511步频谱图。  每个步骤都是一个 101 个单位的向量。
* 输出1375步输出
* 该输出经过多层进一步处理，得到最终的 $T_y = 1375$ 步骤输出。 
* 这个 1D 卷积层的作用类似于你在课程 4 中看到的 2D 卷积，提取低级特征，然后可能生成较小维度的输出。 
* 从计算量来看，一维卷积层还能加快模型运行：GRU 只需处理 1375 个时间步，而非原来的 5511 个时间步。

##### GRU、全连接层与 sigmoid
* 两个 GRU 层从左到右读取输入序列。
* 全连接层加上 sigmoid 层，用于预测 $y^{\langle t \rangle}$。 
* 因为$y$是一个二进制值（0或1），所以我们在最后一层使用sigmoid输出来估计输出为1的机会，对应于用户刚刚说的“activate”。

#### 单向 RNN
* 请注意，我们使用**单向 RNN** 而不是双向 RNN。 
* 这是因为我们认为对于检测激活值单词来说，关注过去的上下文比未来的上下文更重要。这种简化减少了计算开销，因为该模型不会在两个方向上处理信息。 
* 在实际应用中，可以使用滑动的 10 秒窗口每秒执行一次触发词检测。因此，即使激活值这个词可能出现在这10秒内的任何时刻，也可以在说出时立即被检测到。如果算法最初未能检测到它，则随后每一秒的进一步检测都会提供额外的识别机会。这种方法有效地平衡了性能与及时检测尝试。

#### 实现模型

在下面的模型中，每一层的输入都是前一层的输出。实现该模型可以分四个步骤完成。 

**第1步**：CONV层。使用 `Conv1D()` 来实现这一点，有 196 个卷积核， 
卷积核大小为 15 (`kernel_size=15`)，步长为 4。[conv1d](https://keras.io/layers/convolutional/#conv1d)

```Python
output_x = Conv1D(filters=...,kernel_size=...,strides=...)(input_x)
```
* 按照此操作 Batch Normalization。  无需设置任何参数。

```Python
output_x = BatchNormalization()(input_x)
```
* 在此之后添加 ReLU 激活函数。  请注意，我们可以将所需的激活值的名称作为字符串传入，全部为小写字母。

```Python
output_x = Activation("...")(input_x)
```

* 遵循 Dropout，使用比率 0.8 

```Python
output_x = Dropout(rate=...)(input_x)
```


**第2步**：第一个GRU层。要生成 GRU 层，请使用 128 个单元。
```Python
output_x = GRU(units=..., return_sequences = ...)(input_x)
```
* 返回序列，而不仅仅是最后一个时间步的预测，以确保所有 GRU 的隐藏状态都被馈送到下一层。 
* 遵循 Dropout，使用比率 0.8。
* 按照 Batch Normalization 进行操作。  无需设置任何参数。
```Python
output_x = BatchNormalization()(input_x)
```

**步骤 3**：第二个 GRU 层。它具有与第一 GRU 层相同的规格。
* 接下来是 Dropout、Batch Normalization，然后是另一个 Dropout。

**步骤4**：创建时间分布的全连接层，如下所示： 
```Python
output_x = TimeDistributed(Dense(1, activation = "sigmoid"))(input_x)
```
这将创建一个全连接层，后跟一个sigmoid，以便用于全连接层的参数对于每个时间步都是相同的。  
文档：
* [Keras 包装器文档](https://keras.io/layers/wrappers/).  
* 要了解更多信息，你可以阅读这篇博文 [如何在Keras中使用TimeDistributed Layer](https://machinelearningmastery.com/timedistributed-layer-for-long-short-term-memory-networks-in-python/)。

<a name='ex-5'></a>
### 练习 5 - modelf

实现`modelf()`，架构如图3所示。

In [ ]:
# UNQ_C5
# GRADED FUNCTION: modelf

def modelf(input_shape):
    """
    函数在 Keras 中创建模型图。
    
    论证：
    input_shape -- 模型输入数据的形状（使用 Keras 约定）

    返回：
    model -- Keras 模型实例
    """
    
    X_input = Input(shape = input_shape)
    
    ### START CODE HERE ###
    
    # 步骤1：CONV层（约4行）
    # 添加一个包含 196 个单元、内核大小为 15、步长为 4 的 Conv1D
    X = None
    # Batch Normalization
    X = None
    # ReLU 激活函数
    X = None
    # Dropout（使用0.8）
    X = None                                  

    # 步骤 2：第一个 GRU 层（约 4 行）
    # GRU（使用128个单元并返回序列）
    X = None
    # Dropout（使用0.8）
    X = None
    # Batch Normalization.
    X = None                           
    
    # 步骤 3：第二个 GRU 层（约 4 行）
    # GRU（使用128个单元并返回序列）
    X = None
    # Dropout（使用0.8）
    X = None       
    # Batch Normalization
    X = None
    # Dropout（使用0.8）
    X = None                                 
    
    # 步骤 4：时间分布全连接层（≈1 行）
    # 使用 sigmoid 激活值的 TimeDistributed 层 
    X = None 

    ### END CODE HERE ###

    model = Model(inputs = X_input, outputs = X)
    
    return model  

In [ ]:
### 该单元格不可编辑

# 单元测试
from test_utils import *

def modelf_test(target):
    Tx = 5511
    n_freq = 101
    model = target(input_shape = (Tx, n_freq))
    expected_model = [['InputLayer', [(None, 5511, 101)], 0],
                     ['Conv1D', (None, 1375, 196), 297136, 'valid', 'linear', (4,), (15,), 'GlorotUniform'],
                     ['BatchNormalization', (None, 1375, 196), 784],
                     ['Activation', (None, 1375, 196), 0],
                     ['Dropout', (None, 1375, 196), 0, 0.8],
                     ['GRU', (None, 1375, 128), 125184, True],
                     ['Dropout', (None, 1375, 128), 0, 0.8],
                     ['BatchNormalization', (None, 1375, 128), 512],
                     ['GRU', (None, 1375, 128), 99072, True],
                     ['Dropout', (None, 1375, 128), 0, 0.8],
                     ['BatchNormalization', (None, 1375, 128), 512],
                     ['Dropout', (None, 1375, 128), 0, 0.8],
                     ['TimeDistributed', (None, 1375, 1), 129, 'sigmoid']]
    comparator(summary(model), expected_model)
    
    
modelf_test(modelf)

In [ ]:
model = modelf(input_shape = (Tx, n_freq))

让我们打印模型摘要以跟踪形状。

In [ ]:
model.summary()

**预期输出**：

<table>
    <tr>
        <td>
            <strong>总参数</strong>
        </td>
        <td>
           523,329
        </td>
    </tr>
    <tr>
        <td>
            <strong>可训练参数</strong>
        </td>
        <td>
           522,425
        </td>
    </tr>
    <tr>
        <td>
            <strong>不可训练的参数</strong>
        </td>
        <td>
           904
        </td>
    </tr>
</table>

网络的输出形状为 (None, 1375, 1)，而输入形状为 (None, 5511, 101)。 Conv1D 将步数从 5511 减少到 1375。 

<a name='2-2'></a>
### 2.2 - 拟合模型

* 触发词检测需要很长时间来训练。 
* 为了节省时间，我们已经使用你在上面构建的架构以及包含约 4000 个示例的大型训练集在 GPU 上训练了大约 3 个小时的模型。 
* 让我们加载模型。 

In [ ]:
from tensorflow.keras.models import model_from_json

json_file = open('./models/model.json', 'r')
loaded_model_json = json_file.read()
json_file.close()
model = model_from_json(loaded_model_json)
model.load_weights('./models/model.h5')

<a name='2-2-1'></a>
#### 2.2.1 禁用 BatchNormalization 层的训练

如果你要微调预训练模型，则阻止所有批量归一化层的权重非常重要。如果你要从头开始训练新模型，请跳过下一个单元格。 

In [ ]:
model.layers[2].trainable = False
model.layers[7].trainable = False
model.layers[10].trainable = False

你可以使用 Adam 优化器和二元交叉熵损失进一步训练模型，如下所示。这会运行得很快，因为我们只训练两个 epoch，并使用包含 32 个示例的小型训练集。 

In [ ]:
opt = Adam(lr=1e-6, beta_1=0.9, beta_2=0.999)
model.compile(loss='binary_crossentropy', optimizer=opt, metrics=["accuracy"])

In [ ]:
model.fit(X, Y, batch_size = 16, epochs=1)

<a name='2-3'></a>
### 2.3 - 测试模型

最后，让我们看看你的模型在开发集上的表现如何。

In [ ]:
loss, acc, = model.evaluate(X_dev, Y_dev)
print("Dev set accuracy = ", acc)

这看起来很不错！ 
* 然而，准确性并不是这项任务的一个很好的衡量标准
    * 由于标签严重偏向 0，因此仅输出 0 的神经网络将获得略高于 90% 的准确率。 
* 我们可以定义更有用的指标，例如 F1 分数或精确率/召回率。 
    * 让我们在这里不关心这个，而只是凭经验看看模型如何进行一些预测。

<a name='3'></a>
## 3 - 做出预测

现在你已经构建了触发词检测的工作模型，让我们用它来进行预测。此代码片段通过网络运行音频（保存在 wav 文件中）。 

<!--
可以使用你的模型对新的音频片段进行预测。

你首先需要计算输入音频片段的预测。

**练习**：实现predict_activates()。你需要执行以下操作：

1. 计算音频文件的频谱图
2. 使用 `np.swap` 和 `np.expand_dims` 将输入重塑为大小 (1, Tx, n_freqs)
5. 在模型上使用前向传播来计算每个输出步骤的预测
!-->

In [ ]:
def detect_triggerword(filename):
    plt.subplot(2, 1, 1)
    
    # 在预测前校正输入文件的幅度 
    audio_clip = AudioSegment.from_wav(filename)
    audio_clip = match_target_amplitude(audio_clip, -20.0)
    file_handle = audio_clip.export("tmp.wav", format="wav")
    filename = "tmp.wav"

    x = graph_spectrogram(filename)
    # 频谱图输出 (freqs, Tx)，我们希望将 (Tx, freqs) 输入到模型中
    x  = x.swapaxes(0,1)
    x = np.expand_dims(x, axis=0)
    predictions = model.predict(x)
    
    plt.subplot(2, 1, 2)
    plt.plot(predictions[0,:,0])
    plt.ylabel('probability')
    plt.show()
    return predictions

#### 插入提示音以响应“activate”触发词
* 一旦你估计出在每个输出步骤检测到单词“activate”的概率，你就可以在概率高于特定阈值时触发播放“鸣响”声音。 
* 在说“activate”后，$y^{\langle t \rangle}$ 可能连续许多值都接近 1，但我们只想响一次。 
    * 所以我们每75个输出步最多插入一次铃声。 
    * 这将有助于防止我们为单个“activate”实例插入两个提示音。 
    * 这起到了类似于计算机视觉中的非极大值抑制的作用。

<!-- 
**练习**：实现chime_on_activate()。你需要执行以下操作：

1. 循环每个输出步骤的预测概率
2. 当预测大于阈值并且经过超过 75 个连续时间步时，在原始音频片段上插入“钟声”声音

使用此代码将 1,375 步离散化转换为 10,000 步离散化，并使用 pydub 插入“铃声”：

` audio_clip = audio_clip.overlay(chime, 位置 = ((i / Ty) * audio.duration_seconds)*1000)
`
!--> 

In [ ]:
chime_file = "audio_examples/chime.wav"
def chime_on_activate(filename, predictions, threshold):
    audio_clip = AudioSegment.from_wav(filename)
    chime = AudioSegment.from_wav(chime_file)
    Ty = predictions.shape[1]
    # 步骤1：将连续输出步数初始化为0
    consecutive_timesteps = 0
    i = 0
    # 步骤 2：循环 y 中的输出步骤
    while i < Ty:
        # 步骤 3：增加连续输出步骤
        consecutive_timesteps += 1
        # 步骤 4：如果预测高于阈值，则连续 20 个输出步骤已通过
        if consecutive_timesteps > 20:
            # 步骤5：使用pydub叠加音频和背景
            audio_clip = audio_clip.overlay(chime, position = ((i / Ty) * audio_clip.duration_seconds) * 1000)
            # 步骤 6：将连续输出步骤重置为 0
            consecutive_timesteps = 0
            i = 75 * (i // 75 + 1)
            continue
        # 如果幅度小于阈值，则重置 consecutive_timesteps 计数器
        if predictions[0, i, 0] < threshold:
            consecutive_timesteps = 0
        i += 1
        
    audio_clip.export("chime_output.wav", format='wav')

<a name='3-1'></a>
### 3.1 - 开发示例测试

让我们探索我们的模型如何在开发集中的两个未见过的音频片段上执行。让我们先听一下两个开发集片段。 

In [ ]:
IPython.display.Audio("./raw_data/dev/1.wav")

In [ ]:
IPython.display.Audio("./raw_data/dev/2.wav")

现在，在这些音频片段上运行模型，看看它是否会在“activate”之后添加提示音！

In [ ]:
filename = "./raw_data/dev/1.wav"
prediction = detect_triggerword(filename)
chime_on_activate(filename, prediction, 0.5)
IPython.display.Audio("./chime_output.wav")

In [ ]:
filename  = "./raw_data/dev/2.wav"
prediction = detect_triggerword(filename)
chime_on_activate(filename, prediction, 0.5)
IPython.display.Audio("./chime_output.wav")

### 恭喜 

作业你已经结束了！ 

#### 你应该记住以下内容：
- 数据合成是针对语音问题（特别是触发词检测）创建大型训练集的有效方法。 
- 在将音频数据传递到 RNN、GRU 或 LSTM 之前，使用频谱图和可选的 1D 转换层是常见的预处理步骤。
- 端到端深度学习方法可用于构建非常有效的触发词检测系统。 

*恭喜*你完成这份作业！

<a name='4'></a>
## 4 - 尝试你自己的例子！ （可选/不计分）

在 Notebook 的这个可选且不计分的部分中，你可以在自己的音频片段上尝试你的模型！ 

* 录制你说出“activate”一词和其他随机单词的 10 秒音频片段，并将其上传到 Coursera 中心，名称为 `myaudio.wav`。 
* 请务必将音频上传为 wav 文件。 
* 如果你的音频以不同的格式录制（例如 mp3），你可以在线找到免费软件将其转换为 wav。 
* 如果你的录音不是 10 秒，下面的代码将根据需要修剪或填充它以使其达到 10 秒。 

In [ ]:
# 预处理音频，使其符合所需格式
def preprocess_audio(filename):
    # 将音频片段修剪或填充至 10000ms
    padding = AudioSegment.silent(duration=10000)
    segment = AudioSegment.from_wav(filename)[:10000]
    segment = padding.overlay(segment)
    # 将帧速率设置为 44100
    segment = segment.set_frame_rate(44100)
    # 导出为 wav
    segment.export(filename, format='wav')

将音频文件上传到 Coursera 后，将文件的路径放入下面的变量中。

In [ ]:
your_filename = "audio_examples/my_audio.wav"

In [ ]:
preprocess_audio(your_filename)
IPython.display.Audio(your_filename) # 聆听你上传的音频 

最后，使用模型来预测你何时在 10 秒音频片段中说激活并触发提示音。如果未正确添加蜂鸣声，请尝试调整 chime_threshold。

In [ ]:
chime_threshold = 0.5
prediction = detect_triggerword(your_filename)
chime_on_activate(your_filename, prediction, chime_threshold)
IPython.display.Audio("./chime_output.wav")